In [ ]:
%%writefile test_cuda.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void matrixMulKernel(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; ++k) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

void multiply_cuda(const float* h_A, const float* h_B, float* h_C, int N) {
    size_t size = N * N * sizeof(float);
    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(16, 16);
    dim3 blocksPerGrid((N + 15) / 16, (N + 15) / 16);

    matrixMulKernel<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

int main() {
    int N = 1024;
    std::cout << "Allocating memory for CUDA test..." << std::endl;

    float* h_A = new float[N * N];
    float* h_B = new float[N * N];
    float* h_C = new float[N * N];

    for(int i = 0; i < N * N; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
        h_C[i] = 0.0f;
    }

    std::cout << "Running CUDA Matrix Multiplication..." << std::endl;
    multiply_cuda(h_A, h_B, h_C, N);
    std::cout << "Done! Test value at C[0]: " << h_C[0] << " (Should be 2048)" << std::endl;

    delete[] h_A; delete[] h_B; delete[] h_C;
    return 0;
}

Overwriting test_cuda.cu


In [ ]:
!nvcc -O3 test_cuda.cu -o test_cuda && ./test_cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Allocating memory for CUDA test...
Running CUDA Matrix Multiplication...
Done! Test value at C[0]: 2048 (Should be 2048)


In [ ]:
%%writefile test_omp.cpp
#include <iostream>
#include <omp.h>

void multiply_omp(const float* A, const float* B, float* C, int N) {
    #pragma omp parallel for collapse(2)
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            float sum = 0.0f;
            for (int k = 0; k < N; ++k) {
                sum += A[i * N + k] * B[k * N + j];
            }
            C[i * N + j] = sum;
        }
    }
}

int main() {
    int N = 1024;
    std::cout << "Allocating memory for OpenMP test..." << std::endl;

    float* A = new float[N * N];
    float* B = new float[N * N];
    float* C = new float[N * N];

    for(int i = 0; i < N * N; i++) {
        A[i] = 1.0f; B[i] = 2.0f; C[i] = 0.0f;
    }

    std::cout << "Running OpenMP Matrix Multiplication..." << std::endl;
    multiply_omp(A, B, C, N);
    std::cout << "Done! Test value at C[0]: " << C[0] << " (Should be 2048)" << std::endl;

    delete[] A; delete[] B; delete[] C;
    return 0;
}

Overwriting test_omp.cpp


In [ ]:
!g++ -O3 -fopenmp test_omp.cpp -o test_omp && ./test_omp

Allocating memory for OpenMP test...
Running OpenMP Matrix Multiplication...
Done! Test value at C[0]: 2048 (Should be 2048)
